## Chapter 2: Working with Text Data

### 2.2 Tokenizing text

Import text for training model and split text on special characters

In [1]:
# import urllib.request
# url = ("https://raw.githubusercontent.com/rasbt/"
#        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
#        "the-verdict.txt")
# file_path = "the-verdict.txt"
# urllib.request.urlretrieve(url, file_path)


In [2]:
# Read text
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

print(f"Length of text is: {len(raw_text)}")
print(raw_text[:99])

Length of text is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
# Split text on special characters
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### 2.3 Converting tokens into token IDs

Convert all unique tokens into corresponding token IDs and create a vocabulary

In [4]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [5]:
vocab = {token:integer for integer, token in enumerate(all_words)}
# for i, item in enumerate(vocab.items()):
#     print(item)
#     if i >= 20:
#         break

Create class that encodes and decodes

In [6]:
class SimpleTokenizerV1:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        """Process input text into token IDs"""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Converts token IDs back into text"""
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # Remove spaces before specified puntuation
        return text

In [12]:
# Instance new tokenizer

tokenizer = SimpleTokenizerV1(vocab=vocab)
text = "do you like tea?"
ids = tokenizer.encode(text)
print(ids)

[355, 1126, 628, 975, 10]


### 2.4 Adding special context tokens
Adding tokens to handle unknown words (words not found in training data) as well as end of text tokens to seperate texts

In [13]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


Updated tokenizer class

In [14]:
class SimpleTokenizerV2:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        """Process input text into token IDs & replaces unknown word with <|unk|> tokens"""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Converts token IDs back into text"""
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # Remove spaces before specified puntuation
        return text

In [15]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terracesof the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof the palace.


In [16]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text=text))
print(tokenizer.decode(tokenizer.encode(text=text)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 1131, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit <|unk|> the <|unk|>.


### 2.5 Byte pair encoding
More sophisticated tokenizer closer related to what LLMs use

In [17]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [18]:
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [19]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [20]:
# Excercise 2.1 Byte pair encoding of unknown words
words = ("Akwirw ier")
integers = tokenizer.encode(words, allowed_special="all")
print(integers)

[33901, 86, 343, 86, 220, 959]


In [21]:
for i, id in enumerate(integers):
    print(tokenizer.decode(integers[i:i+1]), id)
print(tokenizer.decode(integers))

Ak 33901
w 86
ir 343
w 86
  220
ier 959
Akwirw ier


### 2.6 Data sampling with a sliding window
Generating input-target pairs required for training an LLM

In [22]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [23]:
enc_sample = enc_text[50:]

# Very basic method of creating input-target pairs 
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y;      {y}")

x: [290, 4920, 2241, 287]
y;      [4920, 2241, 287, 257]


In [24]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [25]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [26]:
# Efficient data loader using pytorch
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        """Uses a  sliding window to chunk the text into overlapping sequences of max_length"""
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """Returns total numbers of rows in dataset"""
        return len(self.input_ids)

    def __getitem__(self, index):
        """Returns a single row from the dataset"""
        return self.input_ids[index], self.target_ids[index]


In [29]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                        shuffle=True, drop_last=True, num_workers=0):
    """Function that uses GPTDatasetV1 to load the inputs in batches via a Pytorch DataLoader"""
    tokenizer = tiktoken.get_encoding("gpt2") # Initialize tokenizer
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # Creates dataset
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # Drop last batch if shorter than other batches to prevent loss spikes
        num_workers=num_workers # The number of CPU processes to use for preprocessing
    )

    return dataloader